Step 1: Load Libraries

In [ ]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.linear_model import Ridge  

import pickle

Step 2 : Load data

In [2]:
df = pd.read_csv("steam.csv")
df.head()

,appid,name,release_date,english,developer,publisher,platforms,required_age,categories,genres,steamspy_tags,achievements,positive_ratings,negative_ratings,average_playtime,median_playtime,owners,price
0,10,Counter-Strike,2000-11-01,1,Valve,Valve,windows;mac;linux,0,Multi-player;Online Multi-Player;Local Multi-P...,Action,Action;FPS;Multiplayer,0,124534,3339,17612,317,10000000-20000000,7.19
1,20,Team Fortress Classic,1999-04-01,1,Valve,Valve,windows;mac;linux,0,Multi-player;Online Multi-Player;Local Multi-P...,Action,Action;FPS;Multiplayer,0,3318,633,277,62,5000000-10000000,3.99
2,30,Day of Defeat,2003-05-01,1,Valve,Valve,windows;mac;linux,0,Multi-player;Valve Anti-Cheat enabled,Action,FPS;World War II;Multiplayer,0,3416,398,187,34,5000000-10000000,3.99
3,40,Deathmatch Classic,2001-06-01,1,Valve,Valve,windows;mac;linux,0,Multi-player;Online Multi-Player;Local Multi-P...,Action,Action;FPS;Multiplayer,0,1273,267,258,184,5000000-10000000,3.99
4,50,Half-Life: Opposing Force,1999-11-01,1,Gearbox Software,Valve,windows;mac;linux,0,Single-player;Multi-player;Valve Anti-Cheat en...,Action,FPS;Action;Sci-fi,0,5250,288,624,415,5000000-10000000,3.99


Step 3: Initial data check

In [3]:

df.info()
df.describe()
df.columns


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27075 entries, 0 to 27074
Data columns (total 18 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   appid             27075 non-null  int64  
 1   name              27075 non-null  object 
 2   release_date      27075 non-null  object 
 3   english           27075 non-null  int64  
 4   developer         27074 non-null  object 
 5   publisher         27061 non-null  object 
 6   platforms         27075 non-null  object 
 7   required_age      27075 non-null  int64  
 8   categories        27075 non-null  object 
 9   genres            27075 non-null  object 
 10  steamspy_tags     27075 non-null  object 
 11  achievements      27075 non-null  int64  
 12  positive_ratings  27075 non-null  int64  
 13  negative_ratings  27075 non-null  int64  
 14  average_playtime  27075 non-null  int64  
 15  median_playtime   27075 non-null  int64  
 16  owners            27075 non-null  object

Index(['appid', 'name', 'release_date', 'english', 'developer', 'publisher',
       'platforms', 'required_age', 'categories', 'genres', 'steamspy_tags',
       'achievements', 'positive_ratings', 'negative_ratings',
       'average_playtime', 'median_playtime', 'owners', 'price'],
      dtype='object')

Step 4: Data Cleaning

In [4]:
# Drop duplicates
df = df.drop_duplicates()

# Fill missing values
df['genres'] = df['genres'].fillna('')
df['platforms'] = df['platforms'].fillna('')
df['positive_ratings'] = df['positive_ratings'].fillna(0)
df['negative_ratings'] = df['negative_ratings'].fillna(0)
df['price'] = df['price'].fillna(0)

# Clean text
df['genres'] = df['genres'].apply(lambda x: ';'.join([g.strip() for g in x.split(';') if g]))
df['platforms'] = df['platforms'].apply(lambda x: ';'.join([p.strip() for p in x.split(';') if p]))

Step 5: Create Rating Column

In [5]:
# Calculate rating with division by zero check
df['rating'] = df.apply(
    lambda row: row['positive_ratings'] / (row['positive_ratings'] + row['negative_ratings'])
    if (row['positive_ratings'] + row['negative_ratings']) > 0 else 0,
    axis=1
)

# Scale rating to 0-5
df['rating'] = df['rating'] * 5

Step 6: Convert Price Column

In [15]:
df['price'] = df['price'].astype(float)

Step 7: Feature Engineering

In [6]:
# Popularity feature
df['popularity'] = df['positive_ratings'] + df['negative_ratings']

# Price binning
df['price_bin'] = pd.cut(df['price'], bins=[0,5,20,1000], labels=['cheap','mid','expensive'])

# Combined features for content-based recommendations
df['features'] = df['genres'] + ' ' + df['platforms']

# Genres list for ML
df['genres_list'] = df['genres'].apply(lambda x: x.split(';'))

Step 8: Content-Based Recommendation Setup

In [7]:
tfidf = TfidfVectorizer(
    max_features=1500,     
    stop_words='english',
    ngram_range=(1,1),
    min_df=5
)

tfidf_matrix = tfidf.fit_transform(df['features'])

# Save TF-IDF model
pickle.dump(tfidf, open("assets/tfidf.pkl", "wb"))




Step 9: ML Model Preparation

In [8]:
# Encode genres
mlb = MultiLabelBinarizer()
genres_encoded = mlb.fit_transform(df['genres_list'])

# Feature matrix
X = pd.DataFrame(genres_encoded, columns=mlb.classes_)
X['price'] = df['price']
X['popularity'] = df['popularity']

# Target
y = df['rating']

# ✅ Lightweight model (IMPORTANT)
model = Ridge(alpha=1.0)
model.fit(X, y)

# Save model + encoder
pickle.dump(model, open("assets/rating_model.pkl", "wb"))
pickle.dump(mlb, open("assets/mlb.pkl", "wb"))

Step 10: Save Processed Data

In [9]:
df.to_csv("assets/steam_processed.csv", index=False)